In [0]:
import sys
import os

project_root = os.path.abspath("..")

if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Imports

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

# Paths
OUTPUT_PATH_figures = "/Users/chamika/Desktop/git/machine_learning/Question_1/outputs/figures/"
DATA_PATH_raw    = "/Users/chamika/Desktop/git/machine_learning/Question_1/data/raw/"
DATA_PATH_processed    = "/Users/chamika/Desktop/git/machine_learning/Question_1/data/processed/"

# Setup & Retrain Best Model

In [0]:
# Cell 1: Setup and retrain XGBoost (Tuned) for SHAP analysis

print("=" * 60)
print("SECTION 1.5 — EXPLAINABILITY & ETHICAL CONSIDERATIONS")
print("=" * 60)

# ── Load final dataset ────────────────────────────────────────
df_final = pd.read_csv(DATA_PATH_processed + 'df_final.csv')
print(f"✅ df_final loaded: {df_final.shape}")

# ── Recreate train/test split ─────────────────────────────────
X = df_final.drop(columns=['TransactionID', 'isFraud'])
y = df_final['isFraud']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

print(f"✅ Train/test split: {X_train.shape} / {X_test.shape}")

# ── Retrain XGBoost (Tuned) — best model ─────────────────────
print("\nRetraining XGBoost (Tuned)...")

best_xgb = XGBClassifier(
    n_estimators=484,
    max_depth=8,
    learning_rate=0.0812,
    subsample=0.6911,
    colsample_bytree=0.7297,
    gamma=0.0610,
    min_child_weight=1,
    scale_pos_weight=scale_pos_weight,
    eval_metric='aucpr',
    random_state=42,
    n_jobs=-1,
    verbosity=0
)

best_xgb.fit(X_train, y_train)
print("✅ XGBoost (Tuned) retrained successfully")

# ── Sample for SHAP (SHAP is slow on 118k rows) ───────────────
# 2000 sample is standard practice for SHAP analysis
np.random.seed(42)
sample_idx = np.random.choice(len(X_test), size=2000, replace=False)
X_test_sample = X_test.iloc[sample_idx].reset_index(drop=True)
y_test_sample = y_test.iloc[sample_idx].reset_index(drop=True)

print(f"\n✅ SHAP sample: {X_test_sample.shape}")
print(f"   Fraud rate in sample: {y_test_sample.mean()*100:.2f}%")
print(f"\n✅ Ready for SHAP analysis")

### What is SHAP?
- SHAP = SHapley Additive exPlanations

It answers the question: "Why did the model predict fraud for THIS specific transaction?"

### The Core Idea
Every prediction can be broken down into contributions from each feature:
- Prediction = Base rate + Feature1 contribution + Feature2 contribution + ...

### Why It Matters 

| Reason | Explanation |
|--------|-------------|
| Regulatory compliance | Financial AI must be explainable by law (EU AI Act) |
| Trust | Banks won't deploy a black box — they need to explain decisions to customers |
| Debugging | Reveals if model is learning spurious patterns |
| Ethics | Checks if model is discriminating unfairly |

# Install & Compute SHAP Values

In [0]:
# Cell 2: Install SHAP and compute SHAP value
import shap
print(f"✅ SHAP version: {shap.__version__}")

print("\nComputing SHAP values (1-2 minutes)...")

# TreeExplainer is optimised for XGBoost — very fast
explainer = shap.TreeExplainer(best_xgb)

# Compute SHAP values on sample
shap_values = explainer.shap_values(X_test_sample)

print(f"✅ SHAP values computed")
print(f"   Shape: {shap_values.shape}")
print(f"   Features: {X_test_sample.shape[1]}")
print(f"   Samples:  {X_test_sample.shape[0]}")
print(f"\n   Base value (expected prediction): {explainer.expected_value:.4f}")

# Global feature importance

In [0]:
# Cell 3: Global feature importance — summary (beeswarm) and bar plots
fig = plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_test_sample, show=False, max_display=15)
plt.title('SHAP Summary — Top 15 Features Driving Fraud Predictions', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_PATH_figures + 'shap_summary_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()

fig = plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_test_sample, plot_type='bar', show=False, max_display=15)
plt.title('SHAP Mean |Impact| — Top 15 Features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_PATH_figures + 'shap_summary_bar.png', dpi=150, bbox_inches='tight')
plt.show()

# Print the ranked list too, so we have exact numbers for the report
mean_abs_shap = np.abs(shap_values).mean(axis=0)
shap_importance = pd.Series(mean_abs_shap, index=X_test_sample.columns).sort_values(ascending=False)
print("\n📊 Top 15 features by mean |SHAP value|:")
print(shap_importance.head(15).to_string())

# Individual case explanation

In [0]:
# Cell 4 : Individual case explanation — actual fraud cases
# Ensure alignment between labels and array positions
X_test_sample = X_test_sample.reset_index(drop=True)
y_test_sample = pd.Series(y_test).loc[X_test_sample.index].reset_index(drop=True) if 'y_test_sample' not in dir() else y_test_sample.reset_index(drop=True)

fraud_positions = X_test_sample[y_test_sample == 1].index.tolist()
print("Fraud case positions in SHAP sample:", fraud_positions[:5])

# Pick an actual fraud case this time
example_idx = fraud_positions[0]
predicted_prob = best_xgb.predict_proba(X_test_sample.iloc[[example_idx]])[0, 1]
print(f"Explaining position {example_idx} — true label: FRAUD, predicted probability: {predicted_prob:.4f}")

shap.plots.waterfall(
    shap.Explanation(values=shap_values[example_idx],
                      base_values=explainer.expected_value,
                      data=X_test_sample.iloc[example_idx],
                      feature_names=X_test_sample.columns.tolist()),
    max_display=12, show=False)
plt.title(f'SHAP Waterfall — Correctly Caught Fraud Case (idx {example_idx})', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_PATH_figures + 'shap_waterfall_fraud_caught.png', dpi=150, bbox_inches='tight')
plt.show()